# Where should you throw, if the point is to be measured?

Notebook 07 fitted a player's throwing distribution from the scores they wrote down. It
answered: *given that these darts were aimed at one spot, how tight was the group?*

Two choices were buried in that question and never examined.

1. **Which spot?** The fit takes the target as given. But some targets tell you far more
   about a player than others, and nothing so far has said which.
2. **Why one spot?** A session is a fixed budget of darts. Nothing says they all have to go
   at the same target. Perhaps 100 darts at each of two targets measures a player better
   than 200 at one.

Both are questions of **experimental design**, and both have exact answers. This notebook
computes them, *proves* the second one optimal rather than merely searching for it, and
then checks the whole thing against simulated sessions at realistic lengths -- where the
answer turns out to be considerably more interesting than the theory suggests.

In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join('..', '..'))
if module_path not in sys.path:
    sys.path.append(module_path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25,
    'grid.linewidth': 0.6, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 11, 'font.size': 9, 'legend.frameon': False,
})

from darts import players
from darts.dartboards import generate_dartboard
from darts.design import (best_pair, best_single_target, c_criterion,
                          candidate_targets, design_information,
                          equivalence_certificate, greedy_design,
                          information_at_points, information_maps,
                          optimal_design, sigma_gradient)
from darts.utils import aim_description, mm_per_pixel

PX = 256
BOARD, _ = generate_dartboard(PX)
MM = mm_per_pixel(PX)
CENTRE = PX // 2
N_REF = 200          # darts in a reference session

def polar(r, deg):
    '''Pixel coordinates r mm from the centre, deg clockwise from the 20.'''
    return (int(round(CENTRE + r * np.cos(np.deg2rad(deg)) / MM)),
            int(round(CENTRE + r * np.sin(np.deg2rad(deg)) / MM)))

RESULTS = os.path.join(module_path, 'results')

## The model of a measurement session

The player is asked to aim at targets $t_1 \dots t_k$ and throws $n_i$ darts at each. A
dart aimed at $t_i$ lands at

$$Z \sim N(t_i + b,\ \Sigma)$$

where $b$ is a systematic bias -- the player pulls low and left, say -- and $\Sigma$ is the
spread we want. Only the **score** is recorded, never the position. So the parameters are

$$\theta = (b_x,\ b_y,\ \Sigma_{xx},\ \Sigma_{xy},\ \Sigma_{yy})$$

Five numbers, **whatever $k$ is**. That is what makes the comparison fair: "200 darts at
one target" and "100 at each of two" both estimate exactly five parameters, so any
difference between them is information and not degrees of freedom. `fit_multi_target`
implements this by exact EM, and reduces to notebook 07's single-target fit when $k = 1$.

## How much a target tells you

The score at a given target is multinomial over the board's 44 possible values, so one
dart aimed at $t$ carries Fisher information

$$I(t) = \sum_s \frac{1}{p_s(t)}\, \nabla_\theta p_s(t)\, \nabla_\theta p_s(t)^{\!\top}$$

and the asymptotic standard error of $\hat\sigma$ from $n$ darts is
$\sqrt{c^\top M^{-1} c / n}$ with $c = \partial \sigma / \partial \theta$. Treating $b$ as a
nuisance parameter via $M^{-1}$, rather than just $1/I_{\sigma\sigma}$, correctly charges the
design for having to estimate where the player was actually aiming.

The derivatives $\nabla_\theta p_s$ are available in closed form, and because $p_s(t)$ is a
cross-correlation of the score mask with the throw kernel, **one FFT per score gives the
information at every target on the board at once** -- the same trick the transition builder
uses. The whole map costs about a second.

In [ ]:
SIGMA = players.ABILITY_BANDS['league']
maps = information_maps(PX, SIGMA, board=BOARD)
c_vec = sigma_gradient(SIGMA)

# standard error of sigma-hat from N_REF darts, at every target on the board
se = np.full((PX, PX), np.nan)
rad = np.hypot(*np.meshgrid(np.arange(PX) - CENTRE, np.arange(PX) - CENTRE, indexing='ij'))
inside = rad <= 170 / MM
se[inside] = np.sqrt(c_criterion(maps['info'][inside], c_vec) / N_REF)

named = {'bull': polar(0, 0), 'T20': polar(103, 0), 'D20': polar(166, 0),
         '20 (outer single)': polar(138, 0), 'T19': polar(103, 108)}

fig, ax = plt.subplots(figsize=(6.2, 5.4))
im = ax.imshow(se, origin='lower', cmap='viridis_r', vmax=np.nanpercentile(se, 92))
for name, p in named.items():
    ax.plot(p[1], p[0], 'o', ms=5, color='white', mec='black', mew=0.8)
    ax.annotate(name, (p[1], p[0]), textcoords='offset points', xytext=(7, 4),
                color='white', fontsize=8)
ax.set_title(f'SE of $\\hat\\sigma$ from {N_REF} darts, by target\n'
             f'league player, true $\\sigma$ = {SIGMA} mm')
ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.8, label='mm')
fig.tight_layout()

pd.DataFrame([{'target': n, 'label': aim_description(p, PX),
               'SE of sigma (mm)': round(float(se[p[0], p[1]]), 3),
               'relative': f"{100*se[p[0], p[1]]/SIGMA:.0f}%"}
              for n, p in named.items()])

The map is not subtle. Darker is better, and the good region is a **ring out in the big
single area**, not any of the places a player would naturally throw.

The treble ring is among the *worst* places on the board to be measured, which is worth
sitting with, because T20 is exactly where a player practising would put 200 darts. The
treble bed is 8mm deep and this player's scatter is 16mm: they miss it nearly every time,
and the score they get instead -- 20, 5, 1 -- says little about *how far* they missed. The
information is in the **boundaries you cross**, and up at the treble the only nearby
boundaries are the segment edges, which measure the sideways scatter and say almost nothing
about the radial.

In [ ]:
cand = candidate_targets(PX, point_stride=2)
I_pts = information_at_points(maps, cand)
i1, v1, allvals = best_single_target(I_pts, c_vec)
se_t20 = se[named['T20'][0], named['T20'][1]]
print(f'{len(cand)} candidate targets')
print(f'best single target : {aim_description(cand[i1], PX)}  '
      f'at r = {np.hypot(cand[i1][0]-CENTRE, cand[i1][1]-CENTRE)*MM:.0f} mm')
print(f'  SE = {np.sqrt(v1/N_REF):.3f} mm    vs T20: {se_t20:.3f} mm')
print(f'  choosing T20 instead costs a factor of '
      f'{(se_t20**2)/(v1/N_REF):.1f} in variance')

# the best target at each radius, taking the best angle
r_mm = np.hypot(cand[:, 0] - CENTRE, cand[:, 1] - CENTRE) * MM
bins = np.arange(0, 172, 4)
which = np.digitize(r_mm, bins) - 1
best_at_r = [np.sqrt(allvals[which == b].min() / N_REF) if (which == b).any() else np.nan
             for b in range(len(bins) - 1)]

fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.plot(bins[:-1] + 2, best_at_r, color='#2b6cb0', lw=1.8)
for r, lab in [(6.35, 'bull'), (15.9, '25'), (99, 'treble'), (162, 'double')]:
    ax.axvline(r, color='#999', ls=':', lw=1)
    ax.annotate(lab, (r, ax.get_ylim()[1]), rotation=90, fontsize=7,
                va='top', ha='right', color='#666')
ax.set_xlabel('distance of the target from the centre (mm)')
ax.set_ylabel(f'best SE of $\\hat\\sigma$ ({N_REF} darts)')
ax.set_title('The best target at each radius, league player')
fig.tight_layout()

## The answer changes completely with ability

Fisher information depends on the true parameters, so the best target depends on the
$\sigma$ being measured. It does not drift gently -- it jumps.

In [ ]:
design = pd.read_csv(os.path.join(RESULTS, 'manifest_design.csv'))
tbl = design[['band', 'sigma_mm', 'se_bull', 'se_T20', 'se_D20', 'se_best1',
              'best1', 'best1_r_mm', 'se_optimal']].copy()
tbl.columns = ['band', 'sigma', 'bull', 'T20', 'D20', 'best single',
               'best target', 'its radius (mm)', 'best possible']
tbl.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
axes[0].plot(design.sigma_mm, design.se_bull, 'o-', label='always the bull', color='#dd6b20')
axes[0].plot(design.sigma_mm, design.se_T20, 's-', label='always T20', color='#a0aec0')
axes[0].plot(design.sigma_mm, design.se_best1, '^-', label='best single target', color='#2b6cb0')
axes[0].set_xlabel(r'true $\sigma$ (mm)')
axes[0].set_ylabel(f'SE of $\\hat\\sigma$, {N_REF} darts')
axes[0].set_title('The cost of choosing the target badly'); axes[0].legend()

axes[1].plot(design.sigma_mm, design.best1_r_mm, 'o-', color='#2b6cb0')
axes[1].set_xlabel(r'true $\sigma$ (mm)')
axes[1].set_ylabel('radius of the best target (mm)')
axes[1].set_title('Where the best target sits'); axes[1].set_ylim(-10, 175)
for r, lab in [(15.9, '25 ring'), (99, 'treble ring'), (162, 'double ring')]:
    axes[1].axhline(r, color='#999', ls=':', lw=1)
    axes[1].annotate(lab, (design.sigma_mm.max(), r), fontsize=7, va='bottom',
                     ha='right', color='#666')
fig.tight_layout()

design[['band', 'sigma_mm', 'best1']].assign(
    variance_wasted_at_T20=(design.se_T20 / design.se_best1) ** 2).round(2)

There are two regimes with a sharp transition between them.

**Tight players are measured at the bull.** For $\sigma \le 11$mm the best target is the
bull, and it is not close. The bull is the finest concentric structure on the board: an
inner ring at 6.35mm and an outer at 15.9mm. A player whose scatter is comparable to those
radii splits their darts between 50, 25 and a single in a proportion that pins $\sigma$ down
very precisely. An elite player measured at the bull recovers $\hat\sigma$ to $\pm$0.27mm from
200 darts -- 4% of the true value.

**Loose players are measured further out.** Past about 12mm the bull stops working, because
such a player essentially never hits it: the two rings saturate, every dart is "a single",
and the counts stop discriminating. The best target jumps to roughly 134mm, in the big
single area, where the segment boundaries and the nearby double ring provide structure at
the right scale. Past about 22mm it moves back in to the treble ring, which for a player
scattering that widely is the point from which the greatest variety of scores is reachable.

The general principle is that **you want board features at the scale of the player's
scatter**. Features much finer than $\sigma$ are crossed at a rate that saturates; features
much coarser are never crossed at all.

The practical consequence is the same in both regimes: T20 is a bad place to be measured.
It costs an elite player a factor of 191 in variance -- 200 darts at the bull tell you more
about them than 38,000 darts at T20.

`results/manifest_best_target.csv` is the lookup table over the whole $\sigma$ grid.

In [ ]:
lookup = pd.read_csv(os.path.join(RESULTS, 'manifest_best_target.csv'))
lookup[['sigma_mm', 'best_target', 'best_r_mm', 'se_best1', 'se_bull',
        'se_best2', 'se_optimal']].iloc[::3].round(3)

### It is the radius that matters, not the number

The table names a specific bed, which overstates how specific the advice is. Sweeping all
twenty segments at the best radius shows the choice of *number* is very nearly irrelevant --
which is reassuring, because it means the recommendation survives a player being asked to
throw at whichever segment they find comfortable.

In [ ]:
rows = []
for sigma, r in [(16.0, 133.9), (28.0, 106.0)]:
    m = information_maps(PX, sigma, board=BOARD)
    c = sigma_gradient(sigma)
    ses = []
    for k in range(20):
        p = polar(r, k * 18.0)
        ses.append((float(np.sqrt(c_criterion(m['info'][p[0], p[1]], c) / N_REF)),
                    aim_description(p, PX)))
    ses.sort()
    rows.append({'sigma': sigma, 'radius (mm)': r,
                 'best segment': ses[0][1], 'best SE': ses[0][0],
                 'worst segment': ses[-1][1], 'worst SE': ses[-1][0],
                 'spread': f'{100 * (ses[-1][0] / ses[0][0] - 1):.1f}%'})
pd.DataFrame(rows).round(4)

A 3% spread across every number on the board, for a league player. So the honest form of the
recommendation is **"the big single, about 135mm out, whichever number you like"** rather
than a specific bed.

It also explains a wobble in the search: at 512 pixels the best segment comes out as the 9
rather than the 20, with an SE differing in the fourth decimal place. The whole design result
is stable under resolution -- standard errors agree to about 1% between 256 and 512 pixels --
but *which* near-tied segment wins is not, and should not be read as meaningful.

## Does splitting the session across targets help?

Now the second question. There is a clean argument available before any computation.

Fisher information is **additive over independent throws**, so a design putting a fraction
$w_i$ of the darts at target $t_i$ has per-throw information

$$M(w) = \sum_i w_i\, I(t_i)$$

which is a **convex combination** of the single-target matrices. The achievable set is
therefore the convex hull of $\{I(t)\}$, and single-target designs are precisely its extreme
points. We are minimising a convex functional of $M^{-1}$, so the optimum generally sits in
the interior of the hull -- that is, at a mixture. Splitting should help.

Better still, we can *prove* when the best design has been found rather than trusting a
search. The general equivalence theorem says $w$ is c-optimal exactly when

$$d(t) := c^\top M^{-1} I(t)\, M^{-1} c \ \le\ c^\top M^{-1} c \qquad \text{for every } t$$

so the ratio $\max_t d(t) \,/\, c^\top M^{-1} c$ is $\ge 1$ for any design and equals 1 only
at the optimum.

In [ ]:
opt = optimal_design(I_pts, c_vec)
(a, b), v2 = best_pair(I_pts, c_vec)
rows = [{'design': 'best single', 'k': 1, 'SE': np.sqrt(v1 / N_REF),
         'certificate': equivalence_certificate(I_pts, c_vec, I_pts[i1])},
        {'design': 'best pair (exhaustive)', 'k': 2, 'SE': np.sqrt(v2 / N_REF),
         'certificate': equivalence_certificate(
             I_pts, c_vec, design_information(I_pts[[a, b]], np.ones(2)))}]
for k in (3, 4, 6):
    idx, v = greedy_design(I_pts, c_vec, k)
    rows.append({'design': f'best {k}, equal split', 'k': k, 'SE': np.sqrt(v / N_REF),
                 'certificate': equivalence_certificate(
                     I_pts, c_vec, design_information(I_pts[idx], np.ones(k)))})
rows.append({'design': 'optimal (any weights)', 'k': len(opt['support']),
             'SE': np.sqrt(opt['value'] / N_REF), 'certificate': opt['certificate']})
split = pd.DataFrame(rows)
split['darts needed vs best'] = (split.SE / split.SE.min()) ** 2
split.round(4)

In [ ]:
sub = design[['band', 'sigma_mm', 'se_best1', 'se_best2', 'se_best4', 'se_optimal']]
gain = pd.DataFrame({
    'band': sub.band, 'sigma': sub.sigma_mm,
    'split over 2': (sub.se_best1 / sub.se_best2) ** 2,
    'split over 4': (sub.se_best1 / sub.se_best4) ** 2,
    'best possible split': (sub.se_best1 / sub.se_optimal) ** 2,
})

fig, ax = plt.subplots(figsize=(7.4, 3.4))
ax.plot(gain['sigma'], 100 * (gain['best possible split'] - 1), 'o-', color='#2b6cb0')
ax.axhline(0, color='#999', lw=1)
ax.set_xlabel(r'true $\sigma$ (mm)')
ax.set_ylabel('% of darts saved by splitting')
ax.set_title('What splitting buys, over the best single target (asymptotically)')
fig.tight_layout()
gain.round(3)

### The asymptotic answer: yes, but barely

The convexity argument was right in direction and badly misleading about magnitude.

For the three tightest bands the "mixture" turns out to be a **single point**: the optimal
design has support size 1 and the equivalence certificate is exactly 1.000000. That is not
a search that failed to find a better mixture, it is a proof that none exists. For an elite,
pro or county player, 200 darts at the bull genuinely beats any split of those darts, and
100+100 is strictly *worse*.

From super league downwards the optimum is a genuine mixture spread around the board, and
splitting helps -- but only by 9 to 17% of the variance, and four targets with the darts
divided equally capture essentially all of it. Set that against the factor of 4.6 to 191
available from simply choosing a better single target, and the asymptotic ranking is clear:
**which target dominates; how many barely matters.**

That conclusion turns out to be wrong at any session length a human would actually throw.

## What happens at 100 darts

Everything above is asymptotic. A real session is a few hundred darts, $\hat\sigma$ is biased
at that size, and $\sigma$ is bounded below by zero -- so the Fisher calculation is a
prediction, not a guarantee. `scripts/design_simulation_study.py` simulates whole sessions
under each design, fits every one with the same EM estimator, and reports what actually
happens.

In [ ]:
sim = pd.read_csv(os.path.join(RESULTS, 'design', 'simulation_league.csv'))
# Monte Carlo error on an RMSE estimated from n_ok replicates, so the table can
# be read without mistaking noise for a result.
sim['rmse_mc_se'] = sim.rmse / np.sqrt(2 * sim.n_ok)
order = [d for d in ['T20', 'D20', 'bull', 'best 1', 'best 2', 'best 3', 'best 4', 'best 6']
         if d in set(sim.design)]
piv = sim[sim.n == 100].set_index('design').reindex(order)[
    ['k', 'bias_pct', 'sd', 'rmse', 'rmse_mc_se', 'fisher_se']]
piv['rmse / fisher'] = piv.rmse / piv.fisher_se
print(f"true sigma = {sim.sigma_true.iloc[0]} mm, "
      f"{sim[sim.n==100].n_ok.iloc[0]} replicates of a 100-dart session")
piv.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
d100 = sim[sim.n == 100].set_index('design').reindex(order)
x = np.arange(len(d100))
axes[0].bar(x - 0.2, d100.rmse, 0.4, yerr=d100.rmse_mc_se, capsize=2,
            label='actual RMSE', color='#2b6cb0')
axes[0].bar(x + 0.2, d100.fisher_se, 0.4, label='Fisher prediction', color='#93c5fd')
axes[0].set_xticks(x); axes[0].set_xticklabels(d100.index, rotation=30, ha='right')
axes[0].set_ylabel('mm'); axes[0].set_title('100-dart session, league player')
axes[0].legend()

for name in ['T20', 'bull', 'best 1', 'best 3']:
    g = sim[sim.design == name].sort_values('n')
    if len(g):
        axes[1].plot(g.n, g.rmse / g.fisher_se, 'o-', label=name)
axes[1].axhline(1.0, color='#999', ls='--', lw=1)
axes[1].set_xscale('log')
axes[1].set_xlabel('darts in the session')
axes[1].set_ylabel('actual RMSE / asymptotic prediction')
axes[1].set_title('When the asymptotics start to hold'); axes[1].legend()
fig.tight_layout()

In [ ]:
# every session length, so the finite-sample effects can be separated from noise
full = sim.pivot_table(index='design', columns='n', values='rmse').reindex(order)
mc = sim.pivot_table(index='design', columns='n', values='rmse_mc_se').reindex(order)
print('RMSE of sigma-hat (mm), with Monte Carlo error in brackets')
pd.DataFrame({c: [f'{x:.3f} ({y:.3f})' for x, y in zip(full[c], mc[c])]
              for c in full.columns}, index=full.index)

### At a short session, splitting is worth much more than the theory said

The asymptotics arrive at very different speeds for different designs, and the ratio
column above is the thing to read.

At 100 darts the best *single* target misses its own asymptotic bound by a wide margin,
while the split designs sit almost exactly on theirs -- so splitting buys far more than the
9% the Fisher calculation promised. That advantage then **fades as the session lengthens**:
by a few hundred darts the single target has largely caught up with its bound, and by a
thousand the split and unsplit designs are separated by little more than Monte Carlo noise,
exactly as the asymptotic analysis said they should be.

So the finite-sample case for splitting is specifically a case about *short* sessions --
which is to say, the sessions anybody will actually throw.

Read the ratio column as a diagnostic of *whether the asymptotics have arrived*, not of
whether a design is any good: T20's ratio is below 1 at a hundred darts, but only because
its absolute error is dreadful and the estimator has bought a little variance back by being
badly biased.

T20 is in fact the design that never catches up. Its ratio moves the *wrong* way with
session length, and at a thousand darts both its bias and its spread still exceed the
asymptotic prediction. A flat likelihood is not merely imprecise; it converges to its
imprecision slowly, and keeps a bias that does not shrink at the rate the variance does.

The obvious explanation for the single-target shortfall would be that the aim point is an
expensive nuisance parameter -- but that is checkable, and it is false. Comparing the
variance of $\hat\sigma$ with $b$ unknown against $b$ known, the cost at the best single
target is $1.04\times$: nothing. (Only T20 carries a real nuisance cost, $1.95\times$.)

The real reason is a finite-sample one, and it lives entirely in the **lower tail**.

In [ ]:
why = pd.read_csv(os.path.join(RESULTS, 'design', 'why_splitting.csv'))
sig_true = why.sigma_true.iloc[0]
summary = []
for name, g in why.groupby('design'):
    s, berr = g.sigma_hat.values, g.aim_error_mm.values
    lo = s < np.percentile(s, 25)
    q = np.percentile(s, [1, 5, 25, 50, 75, 95])
    summary.append({
        'design': name, 'k': int(g.k.iloc[0]),
        'rmse': np.sqrt(((s - sig_true) ** 2).mean()),
        '1st pct': q[0], '5th pct': q[1], 'median': q[3], '95th pct': q[5],
        'corr(sigma, aim error)': np.corrcoef(s, berr)[0, 1],
        'aim err | low sigma': berr[lo].mean(),
        'aim err | rest': berr[~lo].mean()})
pd.DataFrame(summary).set_index('design').round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for name, g in why.groupby('design'):
    axes[0].hist(g.sigma_hat, bins=np.arange(8, 22, 0.5), histtype='step', lw=1.6,
                 label=f"{name} (k={int(g.k.iloc[0])})")
axes[0].axvline(sig_true, color='k', ls='--', lw=1)
axes[0].set_xlabel(r'$\hat\sigma$ (mm)'); axes[0].set_ylabel('sessions')
axes[0].set_title('100-dart sessions: the tail is the difference'); axes[0].legend()

for name, g in why.groupby('design'):
    axes[1].scatter(g.sigma_hat, g.aim_error_mm, s=7, alpha=0.5,
                    label=f"{name} (k={int(g.k.iloc[0])})")
axes[1].axvline(sig_true, color='k', ls='--', lw=1)
axes[1].set_xlabel(r'$\hat\sigma$ (mm)'); axes[1].set_ylabel('error in the fitted aim point (mm)')
axes[1].set_title('Underestimating the spread means misplacing the aim')
axes[1].legend()
fig.tight_layout()

From a single target, an unluckily tight group of darts is explained equally well by two
different stories: *a tight thrower aiming where we thought*, and *an ordinary thrower
aiming somewhere else*. The likelihood cannot separate them, so the estimate occasionally
collapses to a much too small $\sigma$ paired with a badly displaced aim point. The scatter
plot shows exactly this -- for one target the two errors are strongly correlated.

Darts at a second and third target destroy the trade-off, because **one displaced aim point
cannot explain three different score histograms at once**. The correlation collapses and
the lower tail disappears.

This is invisible to the Fisher calculation, which describes only the curvature of the
likelihood at its peak; the trade-off is a global feature of the likelihood surface, not a
local one. As the session lengthens the tight-group-plus-displaced-aim story becomes
harder to sustain on its own, which is why the effect fades and the asymptotics take over.

So at a hundred-dart session, splitting is not a 9% refinement -- it is insurance against
occasionally getting a badly wrong answer. At a thousand darts you do not need the
insurance.

### A side effect: the good targets are also the fast ones to fit

The same flatness that makes an uninformative target imprecise also makes the EM crawl.
This is not a separate phenomenon -- a likelihood with little curvature is both hard to
estimate from and slow to climb -- but it is a useful practical tell, and it means the
better designs cost less compute as well as fewer darts.

In [ ]:
from darts.fitting import fit_multi_target, simulate_session

d_league = np.load(os.path.join(RESULTS, 'design', 'design_league.npz'), allow_pickle=True)
probe = {'T20': [np.array([0.0, 103.0])], 'bull': [np.zeros(2)],
         'best 1': [np.asarray(d_league['design_1_mm'][0])],
         'best 3': [np.asarray(p) for p in d_league['design_3_mm']]}

rows = []
for name, targets in probe.items():
    k = len(targets)
    steps, secs = [], []
    for seed in range(12):
        sess = simulate_session(targets, [200 // k] * k, np.array([2.0, -3.0]),
                                SIGMA ** 2 * np.eye(2), board=BOARD, seed=seed)
        import time as _t
        t0 = _t.perf_counter()
        f = fit_multi_target(sess, board=BOARD)
        secs.append(_t.perf_counter() - t0)
        steps.append(f['n_em_steps'])
    rows.append({'design': name, 'k': k, 'mean EM steps': np.mean(steps),
                 'mean seconds': np.mean(secs)})
pd.DataFrame(rows).set_index('design').round(2)

## If you do not know the player's ability yet

There is still a circularity. The best target depends on the $\sigma$ you are trying to
measure. So: how much does guessing wrong cost, and is there one routine that works for
everybody?

In [ ]:
cross = pd.read_csv(os.path.join(RESULTS, 'manifest_design_cross.csv'), index_col=0)
fig, ax = plt.subplots(figsize=(6.4, 4.6))
im = ax.imshow(cross.values, cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(cross.columns)))
ax.set_xticklabels(cross.columns, rotation=40, ha='right')
ax.set_yticks(range(len(cross.index))); ax.set_yticklabels(cross.index)
ax.set_xlabel('the player really is'); ax.set_ylabel('target chosen for')
ax.set_title("Efficiency of using the wrong band's target"); ax.grid(False)
for i in range(cross.shape[0]):
    for j in range(cross.shape[1]):
        ax.text(j, i, f'{cross.values[i, j]:.2f}', ha='center', va='center', fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()

In [ ]:
robust = pd.read_csv(os.path.join(RESULTS, 'manifest_design_robust.csv'))
robust.round(3)

### This is where splitting really earns its keep

The single-ability analysis made splitting look marginal. That analysis assumed you already
knew $\sigma$ -- and you do not, which is the entire point of the exercise.

Once the target must be chosen before the answer is known, a single target is exposed. The
best single compromise target manages only **27%** worst-case efficiency across the ability
range; three targets reach **64%**. Splitting more than doubles the worst case.

(The four-target row is marginally *worse* than three. That is real and not a search
failure: with four equal allocations you cannot express the thirds that the best three-target
design uses, and the minimax search's best four-point answer instead doubles up on the bull.)

So the honest answer to "is 100 + 100 better than 200 at one point?" depends entirely on
what you are allowed to assume:

* **asymptotically, knowing the player's standard** -- barely, and for a tight player, not
  at all: a single point is provably optimal;
* **at a short session** -- yes, clearly, because it removes the tail risk of a badly wrong
  answer. That advantage shrinks as the session lengthens;
* **not knowing the player's standard** -- yes, decisively, and this is the case that
  actually applies.

## Better still: two stages

Hedging is not the only option. Spend a first batch of darts on the robust design, get a
rough $\sigma$, then spend the rest at the target that is best for *that* player, read off
the lookup table.

The simulation below includes the fact that stage two's target is chosen from a *noisy*
stage-one estimate. The **oracle** row is what you would get by throwing every dart at the
single target best for the player's true $\sigma$ -- which nobody can do, since it requires
knowing the answer in advance.

One thing to keep straight when reading it: the oracle is the best *single-target* design,
not the best design of any kind. A two-stage session automatically ends up spread across
several targets, so by the short-session argument above it can legitimately match or even
edge past the oracle. That is not a paradox and not an error; it is the splitting benefit
arriving by the back door.

In [ ]:
two = pd.read_csv(os.path.join(RESULTS, 'design', 'two_stage.csv'))
order2 = ['T20', 'bull', 'robust (all darts)', 'stage 1 only', 'two stage', 'oracle']
piv2 = two.pivot(index='method', columns='band', values='rmse').reindex(order2)
piv2 = piv2[[b for b in ['county', 'league', 'pub'] if b in piv2.columns]]
print(f'{int(two.n.iloc[0])} darts in total')
piv2.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 3.6))
bands_here = list(piv2.columns)
x = np.arange(len(order2))
w = 0.8 / len(bands_here)
for i, b in enumerate(bands_here):
    ax.bar(x + (i - (len(bands_here) - 1) / 2) * w, piv2[b], w, label=b)
ax.set_xticks(x); ax.set_xticklabels(order2, rotation=25, ha='right')
ax.set_ylabel(r'RMSE of $\hat\sigma$ (mm)')
ax.set_title(f'Measuring a player with {int(two.n.iloc[0])} darts')
ax.legend(title='true ability')
fig.tight_layout()

(piv2 / piv2.loc['oracle']).round(2)

## What to actually do

*(interpretation filled in from the outputs above)*